In [ ]:
import pika
import sys

credentials = pika.PlainCredentials('martin', 'martin00')
parameters =  pika.ConnectionParameters('149.62.71.186', credentials=credentials)
connection = pika.BlockingConnection(parameters)
channel = connection.channel()

channel.exchange_declare(exchange='direct_logs', exchange_type='direct')

result = channel.queue_declare(queue='', exclusive=True)
queue_name = result.method.queue

In [ ]:
channel.queue_bind(exchange='direct_logs', queue=queue_name, routing_key='info')
channel.queue_bind(exchange='direct_logs', queue=queue_name, routing_key='warning')
channel.queue_bind(exchange='direct_logs', queue=queue_name, routing_key='error')

In [ ]:
print('Waiting for logs. To exit press CTRL+C')


def callback(ch, method, properties, body):
    print("Received %r:%r" % (method.routing_key, body.decode()))


channel.basic_consume(
    queue=queue_name, on_message_callback=callback, auto_ack=True)

In [ ]:
channel.start_consuming()

### 1. Connection and Credentials
The code starts by authenticating with a RabbitMQ server located at a specific IP address (`149.62.71.186`). It uses the credentials for a user named `martin`.

### 2. The Direct Exchange
```python
channel.exchange_declare(exchange='direct_logs', exchange_type='direct')
```
An **Exchange** is the message router. A `direct` exchange works by looking at a **Routing Key** (a label) attached to a message. If the label on the message matches the label the consumer is looking for, the message is delivered.



### 3. The Temporary Queue
```python
result = channel.queue_declare(queue='', exclusive=True)
queue_name = result.method.queue
```
Instead of using a named, permanent queue (like "Inbox"), this creates a **temporary, exclusive queue**. 
* **`queue=''`**: RabbitMQ generates a random name for the queue (e.g., `amq.gen-Jz9...`).
* **`exclusive=True`**: Once this script stops running or the connection closes, the queue is deleted automatically. This is perfect for gathering logs in real-time.

### 4. Binding (The "Subscription")
This is the most important part of the script:
```python
channel.queue_bind(exchange='direct_logs', queue=queue_name, routing_key='info')
channel.queue_bind(exchange='direct_logs', queue=queue_name, routing_key='warning')
channel.queue_bind(exchange='direct_logs', queue=queue_name, routing_key='error')
```
The script "binds" its temporary queue to the exchange three times. It's essentially telling the exchange: "If a message comes in with the key `info`, `warning`, OR `error`, send a copy to my queue." If a message comes in labeled `debug`, this script will ignore it.

---

## How it Processes Data

* **The Callback:** The `callback` function is the logic that runs every time a message arrives. It prints the severity level (`method.routing_key`) and the actual message content (`body.decode()`).
* **`basic_consume`:** This attaches the callback function to your specific queue.
* **`auto_ack=True`:** This tells RabbitMQ to consider the message "delivered" the moment it's sent to the script, without waiting for the script to confirm it finished processing.

